# Missing Data: Types, Imputation, and Strategy

Handling missing data is one of the most common practical challenges in ML.
The right strategy depends on **why** data is missing.

## Key Concepts

| Type | Definition | Implication |
|------|-----------|-------------|
| **MCAR** (Missing Completely At Random) | Missingness is independent of observed and unobserved data | Safe to drop; any imputation works |
| **MAR** (Missing At Random) | Missingness depends on **observed** data (e.g., older patients skip a test) | Imputation using observed features is valid |
| **MNAR** (Missing Not At Random) | Missingness depends on the **missing value itself** (e.g., high-income people skip income question) | Most dangerous; requires domain modeling or sensitivity analysis |

**Rule of thumb:** MCAR is rare in practice. MAR is the working assumption for most imputation methods. MNAR requires careful thought.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

np.random.seed(42)

## Create a Dataset with Controlled Missing Values

In [ ]:
# Load real data
housing = fetch_california_housing(as_frame=True)
df_full = housing.frame.copy()
target_col = 'MedHouseVal'

print(f"Shape: {df_full.shape}")
print(f"Columns: {list(df_full.columns)}")
df_full.head()

In [ ]:
def introduce_missing(df, col, frac, mechanism='mcar', depend_col=None):
    """Introduce missing values under different mechanisms."""
    df = df.copy()
    n = len(df)
    n_missing = int(n * frac)
    
    if mechanism == 'mcar':
        # Completely random
        idx = np.random.choice(n, n_missing, replace=False)
    elif mechanism == 'mar':
        # Depends on another observed column (higher values -> more likely missing)
        probs = df[depend_col].rank() / n
        probs = probs / probs.sum()
        idx = np.random.choice(n, n_missing, replace=False, p=probs)
    elif mechanism == 'mnar':
        # Depends on the value itself (higher values more likely missing)
        probs = df[col].rank() / n
        probs = probs / probs.sum()
        idx = np.random.choice(n, n_missing, replace=False, p=probs)
    
    df.loc[df.index[idx], col] = np.nan
    return df

# Introduce 20% missing in MedInc (MCAR) and AveRooms (MAR, depends on HouseAge)
df_missing = df_full.copy()
df_missing = introduce_missing(df_missing, 'MedInc', 0.20, 'mcar')
df_missing = introduce_missing(df_missing, 'AveRooms', 0.15, 'mar', depend_col='HouseAge')
df_missing = introduce_missing(df_missing, 'AveOccup', 0.10, 'mnar')

print("Missing values per column:")
print(df_missing.isnull().sum())

## Visualize Missingness Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) Missingness matrix (subset)
sample = df_missing.iloc[:200]
axes[0].imshow(sample.isnull().values.T, aspect='auto', cmap='Greys', interpolation='nearest')
axes[0].set_yticks(range(len(sample.columns)))
axes[0].set_yticklabels(sample.columns, fontsize=8)
axes[0].set_xlabel('Sample index')
axes[0].set_title('Missingness Matrix (white=missing)')

# 2) Missing fraction per column
miss_frac = df_missing.isnull().mean()
miss_frac[miss_frac > 0].plot.barh(ax=axes[1], color='salmon')
axes[1].set_xlabel('Fraction missing')
axes[1].set_title('Missing Fraction per Column')

# 3) Correlation of missingness indicators
miss_indicators = df_missing.isnull().astype(int)
miss_cols = miss_indicators.loc[:, miss_indicators.sum() > 0]
corr = miss_cols.corr()
im = axes[2].imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
axes[2].set_xticks(range(len(corr.columns)))
axes[2].set_xticklabels(corr.columns, rotation=45, fontsize=8)
axes[2].set_yticks(range(len(corr.columns)))
axes[2].set_yticklabels(corr.columns, fontsize=8)
axes[2].set_title('Missingness Correlation')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

## Imputation Method 1: Mean / Median / Mode

Simplest approach. Replace missing values with a summary statistic.

- **Mean:** good for symmetric distributions (sensitive to outliers)
- **Median:** robust to outliers
- **Mode:** for categorical features

**Drawback:** reduces variance, distorts correlations between features.

In [ ]:
class SimpleImputer:
    """Mean/median/mode imputation from scratch."""
    
    def __init__(self, strategy='mean'):
        self.strategy = strategy
        self.fill_values_ = {}
    
    def fit(self, df):
        for col in df.columns:
            if self.strategy == 'mean':
                self.fill_values_[col] = df[col].mean()
            elif self.strategy == 'median':
                self.fill_values_[col] = df[col].median()
            elif self.strategy == 'mode':
                self.fill_values_[col] = df[col].mode().iloc[0]
        return self
    
    def transform(self, df):
        df = df.copy()
        for col in df.columns:
            if col in self.fill_values_:
                df[col] = df[col].fillna(self.fill_values_[col])
        return df

# Demo
imputer_mean = SimpleImputer(strategy='mean')
imputer_mean.fit(df_missing)
df_mean = imputer_mean.transform(df_missing)

imputer_median = SimpleImputer(strategy='median')
imputer_median.fit(df_missing)
df_median = imputer_median.transform(df_missing)

print("Mean imputation fill values (subset):")
for col in ['MedInc', 'AveRooms', 'AveOccup']:
    print(f"  {col}: mean={imputer_mean.fill_values_[col]:.3f}, median={imputer_median.fill_values_[col]:.3f}")

print(f"\nRemaining nulls after mean imputation: {df_mean.isnull().sum().sum()}")

## Imputation Method 2: KNN Imputation

For each sample with missing values, find the K nearest neighbors (using observed features),
then impute using the weighted average of neighbors' values.

**Pros:** captures local structure. **Cons:** O(n^2) distance computation, sensitive to scaling.

In [ ]:
class KNNImputer:
    """KNN imputation from scratch."""
    
    def __init__(self, k=5):
        self.k = k
        self.data_ = None
    
    def fit(self, df):
        self.data_ = df.copy()
        # Store column means as fallback
        self.col_means_ = df.mean()
        # Store column stds for normalization
        self.col_stds_ = df.std().replace(0, 1)
        return self
    
    def transform(self, df):
        result = df.copy()
        # Normalize for distance computation
        data_norm = (self.data_ - self.col_means_) / self.col_stds_
        
        for i in range(len(result)):
            row = result.iloc[i]
            missing_cols = row.index[row.isnull()]
            if len(missing_cols) == 0:
                continue
            
            observed_cols = row.index[row.notnull()]
            if len(observed_cols) == 0:
                result.iloc[i] = self.col_means_
                continue
            
            # Compute distances using observed features only
            row_norm = (row[observed_cols] - self.col_means_[observed_cols]) / self.col_stds_[observed_cols]
            data_obs = data_norm[observed_cols]
            
            # Only use complete rows for these features
            valid_mask = self.data_[list(missing_cols)].notnull().all(axis=1)
            if valid_mask.sum() < self.k:
                for col in missing_cols:
                    result.iloc[i][col] = self.col_means_[col]
                continue
            
            dists = ((data_obs[valid_mask] - row_norm.values) ** 2).sum(axis=1)
            nearest_idx = dists.nsmallest(self.k).index
            
            for col in missing_cols:
                neighbor_vals = self.data_.loc[nearest_idx, col]
                weights = 1.0 / (dists.loc[nearest_idx] + 1e-8)
                result.at[result.index[i], col] = (neighbor_vals * weights).sum() / weights.sum()
        
        return result

# Demo on a small subset (KNN is expensive)
subset_idx = np.random.choice(len(df_missing), 500, replace=False)
df_small = df_missing.iloc[subset_idx].reset_index(drop=True)

knn_imp = KNNImputer(k=5)
knn_imp.fit(df_small)
df_knn = knn_imp.transform(df_small)

print(f"Before KNN imputation: {df_small.isnull().sum().sum()} missing values")
print(f"After KNN imputation: {df_knn.isnull().sum().sum()} missing values")

## Imputation Method 3: Iterative Imputation (MICE Concept)

**MICE** (Multiple Imputation by Chained Equations):
1. Initialize all missing values (e.g., with column means)
2. For each feature with missing values:
   - Treat it as the target, other features as predictors
   - Train a regression model on observed values
   - Predict missing values
3. Repeat for multiple rounds until convergence

This captures inter-feature relationships much better than univariate methods.

In [ ]:
class IterativeImputer:
    """Simplified MICE-style iterative imputation."""
    
    def __init__(self, max_iter=10, tol=1e-3):
        self.max_iter = max_iter
        self.tol = tol
    
    def fit_transform(self, df):
        result = df.copy()
        missing_mask = df.isnull()
        cols_with_missing = [c for c in df.columns if missing_mask[c].any()]
        
        # Step 1: Initialize with column means
        for col in cols_with_missing:
            result[col] = result[col].fillna(result[col].mean())
        
        # Step 2: Iterate
        for iteration in range(self.max_iter):
            old_values = result.copy()
            
            for col in cols_with_missing:
                # Rows where this col was originally missing
                miss_idx = missing_mask[col]
                if miss_idx.sum() == 0:
                    continue
                
                # Predictor columns = everything except current col
                pred_cols = [c for c in df.columns if c != col]
                
                # Train on rows where this col is observed
                train_mask = ~miss_idx
                X_train = result.loc[train_mask, pred_cols].values
                y_train = df.loc[train_mask, col].values  # Use original observed values
                
                # Predict for missing rows
                X_pred = result.loc[miss_idx, pred_cols].values
                
                model = LinearRegression()
                model.fit(X_train, y_train)
                result.loc[miss_idx, col] = model.predict(X_pred)
            
            # Check convergence
            change = np.sqrt(((result.values - old_values.values) ** 2).mean())
            if change < self.tol:
                print(f"  Converged at iteration {iteration + 1} (change={change:.6f})")
                break
        else:
            print(f"  Reached max iterations ({self.max_iter}), change={change:.6f}")
        
        return result

# Demo
iter_imp = IterativeImputer(max_iter=10)
df_iter = iter_imp.fit_transform(df_missing)

print(f"Remaining nulls: {df_iter.isnull().sum().sum()}")

## Missingness Indicator Features

A simple but often effective technique: add binary columns indicating whether a value was missing.
This is especially useful when missingness itself carries information (MAR/MNAR).

In [ ]:
def add_missingness_indicators(df, cols=None):
    """Add binary indicator columns for missingness."""
    result = df.copy()
    if cols is None:
        cols = [c for c in df.columns if df[c].isnull().any()]
    for col in cols:
        result[f'{col}_missing'] = df[col].isnull().astype(int)
    return result

df_with_indicators = add_missingness_indicators(df_missing)
indicator_cols = [c for c in df_with_indicators.columns if c.endswith('_missing')]
print("Indicator columns added:", indicator_cols)
print("\nIndicator value counts:")
for col in indicator_cols:
    print(f"  {col}: {df_with_indicators[col].sum()} ones")

## Comparison: Impact on Model Performance

Train a linear regression using different imputation strategies and compare RMSE.

In [ ]:
def evaluate_imputation(df_imputed, df_original, target_col='MedHouseVal', name=''):
    """Train a model and evaluate RMSE."""
    feature_cols = [c for c in df_imputed.columns if c != target_col]
    X = df_imputed[feature_cols].values
    y = df_imputed[target_col].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return rmse

# Baseline: no missing data
rmse_full = evaluate_imputation(df_full, df_full, name='Full data')

# Mean imputation
rmse_mean = evaluate_imputation(df_mean, df_full, name='Mean')

# Median imputation
rmse_median = evaluate_imputation(df_median, df_full, name='Median')

# Iterative imputation
rmse_iter = evaluate_imputation(df_iter, df_full, name='Iterative')

# Mean imputation + missingness indicators
df_mean_ind = add_missingness_indicators(df_missing)
imp = SimpleImputer(strategy='mean')
imp.fit(df_mean_ind)
df_mean_ind = imp.transform(df_mean_ind)
rmse_mean_ind = evaluate_imputation(df_mean_ind, df_full, name='Mean+Indicators')

# Listwise deletion (drop rows with any missing)
df_dropped = df_missing.dropna()
rmse_dropped = evaluate_imputation(df_dropped, df_full, name='Listwise deletion')

results = {
    'Full data (oracle)': rmse_full,
    'Listwise deletion': rmse_dropped,
    'Mean imputation': rmse_mean,
    'Median imputation': rmse_median,
    'Iterative (MICE)': rmse_iter,
    'Mean + indicators': rmse_mean_ind
}

print("\nRMSE Comparison:")
print("-" * 40)
for name, rmse in sorted(results.items(), key=lambda x: x[1]):
    print(f"  {name:25s}: {rmse:.4f}")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
values = list(results.values())
colors = ['green' if n == 'Full data (oracle)' else 'steelblue' for n in names]

bars = ax.barh(names, values, color=colors)
ax.set_xlabel('RMSE (lower is better)')
ax.set_title('Imputation Strategy Comparison')
ax.axvline(x=rmse_full, color='green', linestyle='--', alpha=0.5, label='Oracle (no missing)')
ax.legend()

for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')

plt.tight_layout()
plt.show()

## Visualize: Distribution Distortion from Imputation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

col = 'MedInc'
for ax, (label, data) in zip(axes, [
    ('Original', df_full[col]),
    ('Mean imputed', df_mean[col]),
    ('Iterative imputed', df_iter[col])
]):
    ax.hist(data, bins=50, alpha=0.7, density=True, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{label}\nstd={data.std():.3f}')
    ax.set_xlabel(col)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.2f}')
    ax.legend(fontsize=8)

plt.suptitle('Distribution of MedInc under Different Imputation', y=1.02)
plt.tight_layout()
plt.show()

print("Note: Mean imputation creates a spike at the mean, reducing variance.")
print("Iterative imputation better preserves the original distribution shape.")

## Interview Questions

### Q: How do you handle missing data?
**A:** First, understand the mechanism (MCAR/MAR/MNAR). For MCAR with small fraction missing, deletion may be fine. For MAR, model-based imputation (iterative/KNN) works well. For MNAR, add missingness indicators and consider domain-specific approaches. Always compare imputation strategies on validation performance.

### Q: When is deletion OK?
**A:** When: (1) data is MCAR, (2) fraction missing is small (<5%), (3) you have plenty of data. Deletion is problematic when missingness is informative or when it removes a biased subset of data.

### Q: MCAR vs MAR -- why does it matter?
**A:** Under MCAR, the observed data is a random sample of the full data, so simple methods work. Under MAR, the missingness depends on other variables -- you need to condition on those variables during imputation (e.g., iterative imputation). Under MNAR, the missingness depends on the missing value itself, and standard imputation methods will be biased.

### Q: Why add missingness indicators?
**A:** When data is MAR or MNAR, the *fact* that a value is missing carries information. A missingness indicator captures this signal. Example: if income is missing because of high income, the indicator encodes information the imputed value cannot.

### Quick Decision Tree
```
Missing data?
├── < 5% and MCAR → Consider deletion or mean/median imputation
├── Moderate amount → KNN or iterative imputation + missingness indicators
├── High cardinality categorical → Mode or dedicated category "missing"
└── MNAR suspected → Missingness indicators essential, consider domain modeling
```